# Project 1 — Classifying user requests

This notebook follows the order of [Tutorial 1](tutorial.ipynb): **setup → data → text segmentation → representations → classifiers → experiments → analysis**.

Our question is: **how do text representation and classifier architecture affect harmful-versus-unharmful prompt classification?** We use only the prompt text from the agreed filtered WildGuardMix training data.

Run the cells from top to bottom. The explanation sits beside the code it describes. Settings are ordinary Python variables, and the experiment loop is visible here. Small repeated operations live in [`../utils/p1/functions.py`](../utils/p1/functions.py).

If dependencies are missing, run [setup.ipynb](setup.ipynb) first. The original [project assignment](project.ipynb) and tutorial remain unchanged.

## 1. Setup

Like the tutorial, we first make the repository importable and load the libraries. Select the project's `.venv` kernel. We use CPU execution and a fixed seed so the run can be repeated.

In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part1/tutorial.ipynb").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep generated caches in this local, Git-ignored directory.
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(root / ".cache"))
os.environ.setdefault("HF_HUB_CACHE", str(root / ".cache/huggingface/hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:
import hashlib
import json
import random
import time
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

import part1
from utils.p1.functions import (
    TokenIndicators, BagOfWords, TFIDF, encode_texts,
    TwoHiddenMLP, train_mlp, predict_mlp, classification_scores,
)

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

### Settings

For the real experiment we use **1,500 prompts per class, 1,500 BPE tokens and 10 neural-training epochs**. These are manageable starting choices, not tuned optima.

Set `use_toy_data = True` for a quick offline check. Its harmless gardening/astronomy examples have artificial topic labels, so their scores are **not scientific harmfulness results**. Dataset access errors never cause an automatic switch to toy data.

In [ ]:
use_toy_data = False
seed = 42
num_folds = 5
examples_per_class = 2000 #1500
vocab_size = 3000 #1500
min_frequency = 2
num_epochs = 10
hidden_dim = 64
second_hidden_dim = 32
dropout = 0.2
batch_size = 64
learning_rate = 0.001
weight_decay = 0.0001
rope = 0.01

dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
if use_toy_data:
    examples_per_class, vocab_size, num_epochs = 20, 128, 2

run_label = "SMOKE_NOT_SCIENTIFIC" if use_toy_data else "PROJECT1_CV"
results_folder = root / "part1/results" / ("toy" if use_toy_data else "project1")
results_folder.mkdir(parents=True, exist_ok=True)
print(run_label)

## 2. Load the data

We predict `prompt_harm_label` from `prompt` only. The subcategory is used to filter rows, never as a model feature. We do not load response text into our working table.

WildGuardMix requires accepting its terms and a Hugging Face login. The setup notebook explains this. **Only `wildguardtrain/train` is loaded. The official test stays untouched.** We pin the dataset revision and also verify it if Hugging Face uses a local cache.

We display shapes, counts and safe teaching examples throughout this notebook, rather than raw dataset prompts.

In [ ]:
if use_toy_data:
    records = []
    for i in range(examples_per_class):
        records.append({"prompt": f"Garden notebook {i}: water the flowers and prepare fresh soil.",
                        "prompt_harm_label": "unharmful", "subcategory": "synthetic"})
        records.append({"prompt": f"Space notebook {i}: observe the planets through a telescope tonight.",
                        "prompt_harm_label": "harmful", "subcategory": "synthetic"})
    data = pd.DataFrame(records)
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(root / ".cache/huggingface/datasets"),
    )
    assert dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in dataset.cache_files
    ), "The cached dataset does not match our pinned revision."
    data = dataset.select_columns(["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

print("Rows loaded:", len(data))

### 2.1 Apply the agreed category filter first

Remove an exact `strip().casefold()` match to `toxic_language_hate_speech`. Then remove missing harmfulness labels and map `unharmful` to 0 and `harmful` to 1.

The team audit expects **86,759 raw rows → 83,739 eligible rows**, before prompt cleanup. We check those counts instead of silently accepting a different dataset.

In [ ]:
audit = {"status": run_label, "original_rows": len(data)}
excluded = data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech")
audit["excluded_category_rows"] = int(excluded.sum())
data = data.loc[~excluded].copy()
assert not data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech").any()

labels = data["prompt_harm_label"].str.strip().str.casefold()
missing_label = labels.isna() | labels.eq("")
audit["missing_label_rows"] = int(missing_label.sum())
assert labels.loc[~missing_label].isin(["unharmful", "harmful"]).all(), "Unexpected label."
data = data.loc[~missing_label].copy()
data["label"] = labels.loc[~missing_label].map({"unharmful": 0, "harmful": 1}).astype(int)
audit["eligible_rows"] = len(data)
audit["eligible_unharmful"] = int(data["label"].eq(0).sum())
audit["eligible_harmful"] = int(data["label"].eq(1).sum())

if not use_toy_data:
    expected = {"original_rows": 86759, "excluded_category_rows": 3020,
                "missing_label_rows": 0, "eligible_rows": 83739,
                "eligible_unharmful": 40543, "eligible_harmful": 43196}
    assert all(audit[key] == value for key, value in expected.items()), "Dataset audit mismatch."
display(pd.Series(audit, name="Data audit"))

### 2.2 Clean and deduplicate prompts

Normalize Unicode to NFC and strip edge whitespace. Preserve case and internal whitespace. Remove empty text. If the same normalized prompt has conflicting labels, drop the whole group; otherwise keep one copy.

This prevents exact copies of a prompt from appearing in both training and validation. Paraphrases and related scenarios can still cross folds, which remains a limitation. Removal counts below are separate and add up.

In [ ]:
data["prompt"] = data["prompt"].fillna("").str.normalize("NFC").str.strip()
empty_prompt = data["prompt"].eq("")
audit["empty_prompt_rows"] = int(empty_prompt.sum())
data = data.loc[~empty_prompt].copy()

conflicting = data.groupby("prompt")["label"].transform("nunique").gt(1)
audit["contradictory_label_rows"] = int(conflicting.sum())
audit["contradictory_label_groups"] = int(data.loc[conflicting, "prompt"].nunique())
data = data.loc[~conflicting].copy()
audit["duplicate_rows"] = int(data.duplicated("prompt").sum())
data = data.drop_duplicates("prompt").copy()
audit["retained_rows"] = len(data)

removals = ["excluded_category_rows", "missing_label_rows", "empty_prompt_rows",
            "contradictory_label_rows", "duplicate_rows", "retained_rows"]
assert sum(audit[key] for key in removals) == audit["original_rows"]
assert data["prompt"].is_unique

data["id"] = data["prompt"].map(lambda text: "wg-" + hashlib.sha256(text.encode("utf-8")).hexdigest())
data = data[["id", "prompt", "label"]].sort_values("id").reset_index(drop=True)
display(pd.Series(audit, name="Completed audit"))

### 2.3 Take a reproducible sample

A balanced 3,000-example sample keeps the experiment practical on CPU. Its 50% harmful prevalence differs from the original training population (about 51.58% before cleanup). We report both.

The hash IDs let us save and match predictions without exporting prompt text. Hashing is identification, not encryption.

In [ ]:
rng = random.Random(seed)
chosen_indices = []
for label in [0, 1]:
    candidates = data.index[data["label"].eq(label)].tolist()
    assert len(candidates) >= examples_per_class
    chosen_indices.extend(rng.sample(candidates, examples_per_class))
rng.shuffle(chosen_indices)
sample = data.loc[chosen_indices].reset_index(drop=True)

texts = sample["prompt"].tolist()
y = sample["label"].to_numpy()
ids = sample["id"].to_numpy()
audit["sample_rows"] = len(sample)
audit["sample_harmful_prevalence"] = float(y.mean())
print("Sample class counts:", sample["label"].value_counts().sort_index().to_dict())
print("Eligible population harmful prevalence:", audit["eligible_harmful"] / audit["eligible_rows"])
display(sample["prompt"].str.len().describe(percentiles=[0.5, 0.95]).rename("Prompt length in characters"))

## 3. Create five shared folds

We use `StratifiedKFold` to keep both labels represented in every fold. The tutorial's splitting helper shuffles but is not stratified. Every system will reuse this **same partition**.

We create the folds before learning BPE or IDF. Training those on the entire sample would let validation text influence the features.

In [ ]:
splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
folds = list(splitter.split(texts, y))
fold_assignment = np.full(len(sample), -1)
for fold, (train_indices, validation_indices) in enumerate(folds):
    assert set(train_indices).isdisjoint(validation_indices)
    fold_assignment[validation_indices] = fold
    print(f"Fold {fold + 1}: {len(train_indices)} train, {len(validation_indices)} validation")
assert (fold_assignment >= 0).all()

# Start with one fold to understand each component, like the tutorial.
train_indices, validation_indices = folds[0]
train_texts = [texts[i] for i in train_indices]
validation_texts = [texts[i] for i in validation_indices]
y_train, y_validation = y[train_indices], y[validation_indices]

## 4. Digital Text: BPE tokenization

We directly reuse `part1.experiment.train_tokenizer`. It learns BPE merges with a Metaspace pretokenizer and an `[UNK]` token. Fit it on the first fold's **training text only**.

BPE tokens are useful learned units, not necessarily words or linguistic morphemes. The following sentence is a harmless teaching example, not a dataset row.

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
safe_sentence = "Please explain how plants grow in a garden."
print(tokenizer.encode(safe_sentence).tokens)
print("Learned vocabulary size:", tokenizer.get_vocab_size())

## 5. Text Classification: three representations

- **Indicators:** 1 if a BPE token appears, otherwise 0.
- **Bag-of-words:** the number of occurrences of each token. This completes the tutorial's counting exercise.
- **TF-IDF:** counts multiplied by `log((1 + n_train) / (1 + df_train)) + 1`, then L2-normalized.

All three use the same tokenizer within a fold. They lose token order. TF-IDF learns its document frequencies only from the training texts.

`TokenIndicators` reuses the course `TokenIndicator` and fixes its `__len__` method to use `self.tokenizer` instead of a global variable. Other repeated encoders are in the helper file.

In [ ]:
indicator_encoder = TokenIndicators(tokenizer)
bow_encoder = BagOfWords(tokenizer)
tfidf_encoder = TFIDF(tokenizer).fit(train_texts)
encoders = {"indicators": indicator_encoder, "bow": bow_encoder, "tfidf": tfidf_encoder}

safe_repetition = "garden garden garden plants"
token_ids = np.flatnonzero(bow_encoder(safe_repetition))
display(pd.DataFrame({
    "token": [tokenizer.id_to_token(int(i)) for i in token_ids],
    "indicator": indicator_encoder(safe_repetition)[token_ids],
    "count": bow_encoder(safe_repetition)[token_ids],
    "tfidf": tfidf_encoder(safe_repetition)[token_ids],
}))
assert bow_encoder(safe_repetition).max() > 1
assert set(indicator_encoder(safe_repetition)).issubset({0, 1})

### Represent the first fold

For the next three examples, hold the representation fixed at TF-IDF. `encode_texts` applies an encoder to each text and creates the SciPy sparse matrix expected by the course models.

Transforming validation text must not update the tokenizer or IDF. The assertions check that explicitly.

In [ ]:
vocabulary_before = tokenizer.get_vocab().copy()
idf_before = tfidf_encoder.transformer.idf_.copy()
X_train = encode_texts(train_texts, tfidf_encoder)
X_validation = encode_texts(validation_texts, tfidf_encoder)
assert tokenizer.get_vocab() == vocabulary_before
np.testing.assert_array_equal(tfidf_encoder.transformer.idf_, idf_before)
print("Feature matrix shapes:", X_train.shape, X_validation.shape)

## 6. Log-linear classifier

Like `part1.loglin`, we use sklearn logistic regression. Here we pass the feature matrices directly because we have already encoded the texts. This avoids repeating the tutorial wrapper's encoding step. `lbfgs` fits the same regularized logistic model family and works well with these small sparse matrices.

The two probability columns correspond to `[unharmful, harmful]`. The decision rule is the tutorial's most-probable-class rule. An exact tie goes to class 0.

In [ ]:
logistic_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=seed)
logistic_model.fit(X_train, y_train)
logistic_probabilities = logistic_model.predict_proba(X_validation)
logistic_predictions = logistic_probabilities.argmax(axis=1)
first_fold_scores = {"logistic": classification_scores(y_validation, logistic_predictions)}
display(pd.Series(first_fold_scores["logistic"]))

## 7. The tutorial's single-hidden-layer MLP

We instantiate `part1.smlp.SparseMLP` directly: sparse input, 64 ReLU units, dropout and log-softmax output. The reusable training helper calls the course `SparseDataset`, `sparse_collate` and `train_one_epoch`.

We train for a fixed number of epochs. The held-out fold does not choose a checkpoint or stopping epoch. Negative log-likelihood consumes the model's log probabilities.

In [ ]:
torch.manual_seed(seed)
mlp_model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, num_classes=2, dropout=dropout)
training_history = train_mlp(
    mlp_model, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
mlp_probabilities = predict_mlp(mlp_model, X_validation)
first_fold_scores["mlp1"] = classification_scores(y_validation, mlp_probabilities.argmax(axis=1))
display(pd.DataFrame(training_history).rename_axis("epoch"))

## 8. Our extra architecture: two hidden layers

`TwoHiddenMLP` extends the official `SparseMLP`. It keeps its sparse first layer and changes the output module to a second 32-unit ReLU layer followed by the class output.

We keep dropout, optimizer and training budget the same. This changes both architecture and parameter count; it is not a parameter-matched comparison.

In [ ]:
torch.manual_seed(seed)
modified_mlp = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
modified_history = train_mlp(
    modified_mlp, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
modified_probabilities = predict_mlp(modified_mlp, X_validation)
first_fold_scores["mlp2"] = classification_scores(y_validation, modified_probabilities.argmax(axis=1))
display(pd.DataFrame(first_fold_scores).T)
print("These are first-fold demonstrations. The complete comparison comes next.")

## 9. Experiment: all nine systems on the same five folds

Now repeat the same steps for each fold. The loop is intentionally visible:

1. Select training and validation rows.
2. Train a **new BPE tokenizer and IDF** on that training fold.
3. Build indicators, counts and TF-IDF.
4. Fit logistic regression, the one-layer MLP and the two-layer MLP.
5. Score each model on its held-out fold and keep its predictions.

That gives **3 representations × 3 models × 5 folds = 45 fits**. The earlier demonstrations do not enter this table. Macro-F1 is our primary metric. Harmful recall and benign false-positive rate explain different types of mistakes. Fit time below excludes feature construction.

In [ ]:
score_rows = []
prediction_rows = []
model_names = ["logistic", "mlp1", "mlp2"]
representation_names = ["indicators", "bow", "tfidf"]

for fold, (train_indices, validation_indices) in enumerate(folds):
    fold_seed = seed + fold
    train_texts = [texts[i] for i in train_indices]
    validation_texts = [texts[i] for i in validation_indices]
    y_train, y_validation = y[train_indices], y[validation_indices]

    tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
    encoders = {
        "indicators": TokenIndicators(tokenizer),
        "bow": BagOfWords(tokenizer),
        "tfidf": TFIDF(tokenizer).fit(train_texts),
    }
    for representation, encoder in encoders.items():
        X_train = encode_texts(train_texts, encoder)
        X_validation = encode_texts(validation_texts, encoder)

        for model_name in model_names:
            started = time.perf_counter()
            if model_name == "logistic":
                model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=fold_seed)
                model.fit(X_train, y_train)
                fit_seconds = time.perf_counter() - started
                probabilities = model.predict_proba(X_validation)
            else:
                torch.manual_seed(fold_seed)
                if model_name == "mlp1":
                    model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, 2, dropout)
                else:
                    model = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
                history = train_mlp(model, X_train, y_train, epochs=num_epochs, seed=fold_seed,
                                    batch_size=batch_size, learning_rate=learning_rate,
                                    weight_decay=weight_decay)
                fit_seconds = time.perf_counter() - started
                probabilities = predict_mlp(model, X_validation)

            assert np.isfinite(probabilities).all()
            np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
            predictions = probabilities.argmax(axis=1)
            metrics = classification_scores(y_validation, predictions)
            score_rows.append({"status": run_label, "model": model_name,
                               "representation": representation, "fold": fold,
                               **metrics, "fit_seconds": fit_seconds, "seed": fold_seed})
            for j, index in enumerate(validation_indices):
                prediction_rows.append({"id": ids[index], "model": model_name,
                    "representation": representation, "fold": fold, "gold": int(y[index]),
                    "prediction": int(predictions[j]), "confidence": float(probabilities[j].max()),
                    "probability_harmful": float(probabilities[j, 1]), "status": run_label})
    print(f"Finished fold {fold + 1}/{num_folds}")

scores = pd.DataFrame(score_rows)
oof = pd.DataFrame(prediction_rows)
assert len(scores) == 45
assert len(oof) == len(sample) * 9
assert not oof.duplicated(["id", "model", "representation"]).any()
assert oof.groupby("id")["fold"].nunique().eq(1).all()
print(run_label, "— completed all 45 fits")

## 10. Compare the results

Average each system's five held-out scores. The standard deviation describes variation across folds; it is not a confidence interval from independent experiments. Fold training sets overlap.

We reuse the tutorial's `plot_box_and_points`, supplying macro-F1 rather than accuracy.

In [ ]:
summary = scores.groupby(["model", "representation"]).agg(
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    accuracy=("accuracy", "mean"), harmful_recall=("harmful_recall", "mean"),
    benign_fpr=("benign_fpr", "mean"), fit_seconds=("fit_seconds", "mean"),
).reset_index()
display(summary.round(4))

plot_data = scores.assign(model=scores["model"] + "/" + scores["representation"], run=0)
part1.analysis.plot_box_and_points(plot_data, metric="macro_f1")
# The course helper calls plt.show(); the numeric table above remains the source of truth.

## 11. Bayesian comparison and ROPE

We specify two interpretable comparisons before interpreting the results:

1. Logistic **indicators vs TF-IDF**: change representation, keep architecture fixed.
2. TF-IDF **logistic vs one-layer MLP**: change architecture, keep representation fixed.

The tutorial's `BenavoliCorrelatedT` models the difference **B − A**, so positive values favour B. We propose **±0.01 macro-F1** as the region of practical equivalence: a one-percentage-point change alone would not justify added complexity in this small study. This is a project judgment, not a general safety threshold.

Five paired folds provide limited evidence. The posterior depends on the correlated-CV model assumptions. In an artificial toy run, all fold differences may be identical; that zero-variance case is represented by a point mass rather than a zero-scale t distribution.

In [ ]:
comparisons = [
    (("logistic", "indicators"), ("logistic", "tfidf")),
    (("logistic", "tfidf"), ("mlp1", "tfidf")),
]
comparison_rows = []

for system_a, system_b in comparisons:
    a = scores[(scores.model == system_a[0]) & (scores.representation == system_a[1])].sort_values("fold")
    b = scores[(scores.model == system_b[0]) & (scores.representation == system_b[1])].sort_values("fold")
    assert a["fold"].tolist() == b["fold"].tolist()
    differences = b["macro_f1"].to_numpy() - a["macro_f1"].to_numpy()
    names = ["/".join(system_a), "/".join(system_b)]

    if np.allclose(differences, differences[0]):
        mean_difference = float(differences.mean())
        equivalent = abs(mean_difference) <= rope or np.isclose(abs(mean_difference), rope)
        p_a = float(mean_difference < -rope and not equivalent)
        p_tie = float(equivalent)
        p_b = float(mean_difference > rope and not equivalent)
        fig, ax = plt.subplots()
        ax.axvspan(-rope, rope, alpha=0.2, color="teal", label="ROPE")
        ax.axvline(mean_difference, label="Point mass: identical fold differences")
        ax.set_xlabel("Mean macro-F1 difference B - A")
        ax.legend()
    else:
        posterior = part1.analysis.BenavoliCorrelatedT(
            a["macro_f1"].to_numpy(), b["macro_f1"].to_numpy(), runs=1, names=names)
        p_a, p_tie, p_b = posterior.probs(rope)
        mean_difference = float(posterior.mean_diff)
        fig, ax = posterior.plot(rope)
        assert np.isclose(mean_difference, differences.mean())

    ax.set_title(f"{run_label}\nA: {names[0]} vs B: {names[1]}")
    fig.tight_layout()
    fig.savefig(results_folder / f"rope_{len(comparison_rows) + 1}.png", dpi=150, bbox_inches="tight")
    plt.show()
    assert np.isclose(p_a + p_tie + p_b, 1)
    comparison_rows.append({"A": names[0], "B": names[1], "mean_B_minus_A": mean_difference,
                            "rope": rope, "P_A_wins": p_a, "P_equivalent": p_tie, "P_B_wins": p_b})

bayesian_results = pd.DataFrame(comparison_rows)
display(bayesian_results.round(4))

## 12. Error analysis

Select the system with the highest observed mean macro-F1 for descriptive inspection. This ranking alone does not establish that it is decisively better than every alternative.

A **false positive** flags an unharmful prompt. A **false negative** misses a harmful prompt. We also inspect errors assigned at least 0.8 predicted-class probability. These probabilities are uncalibrated, so confidence is not a guarantee of correctness.

The table displays only opaque IDs. Raw prompts remain local and are not exported. Topic vocabulary, prompt length and ambiguous annotation are possible explanations to investigate; the metrics alone cannot establish lexical causes.

In [ ]:
best = summary.loc[summary["macro_f1_mean"].idxmax()]
best_predictions = oof[(oof.model == best["model"]) & (oof.representation == best["representation"])].copy()
print("Descriptive CV leader:", best["model"], best["representation"])
counts = classification_scores(best_predictions["gold"], best_predictions["prediction"])
display(pd.DataFrame([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]],
                     index=["Gold unharmful", "Gold harmful"],
                     columns=["Predicted unharmful", "Predicted harmful"]))

errors = best_predictions[best_predictions.gold != best_predictions.prediction].copy()
errors["error_type"] = np.where(errors.gold == 0, "false_positive", "false_negative")
errors["confidently_wrong"] = errors["confidence"] >= 0.8
display(errors.groupby("error_type").agg(errors=("id", "count"), confidently_wrong=("confidently_wrong", "sum")))
display(errors.sort_values("confidence", ascending=False).head(10))

## 13. Save the experiment

Save scores, anonymous predictions, sample fold assignments, audit counts and settings. No prompt text, learned vocabulary or model checkpoint is exported. Results are local under `part1/results/` and ignored by Git.

When the team is ready to share results, review the aggregate files first. This notebook itself can be saved with outputs cleared. No GitHub publication is performed here.

In [ ]:
scores.to_csv(results_folder / "p1_cv_scores.csv", index=False)
summary.to_csv(results_folder / "p1_summary.csv", index=False)
oof.to_csv(results_folder / "p1_oof_predictions.csv", index=False)
errors.to_csv(results_folder / "p1_errors.csv", index=False)
bayesian_results.to_csv(results_folder / "p1_rope.csv", index=False)
pd.DataFrame({"id": ids, "gold": y, "fold": fold_assignment}).to_csv(results_folder / "folds.csv", index=False)
(results_folder / "data_audit.json").write_text(json.dumps(audit, indent=2))

settings = {"status": run_label, "seed": seed, "folds": num_folds,
            "examples_per_class": examples_per_class, "vocab_size": vocab_size,
            "min_frequency": min_frequency, "epochs": num_epochs,
            "hidden_dim": hidden_dim, "second_hidden_dim": second_hidden_dim,
            "dropout": dropout, "batch_size": batch_size, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "logistic_C": 1.0, "logistic_solver": "lbfgs",
            "rope": rope, "dataset_revision": None if use_toy_data else dataset_revision,
            "official_test_accessed": False,
            "packages": {name: version(name) for name in
                         ["numpy", "pandas", "scipy", "scikit-learn", "torch", "tokenizers", "datasets", "baycomp"]}}
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print("Saved:", results_folder.relative_to(root))

## 14. Questions to discuss together

- What information does BPE preserve, and how are unseen characters handled?
- What changes when counts replace indicators? What does TF-IDF downweight?
- Why do BPE and IDF have to be fitted separately in each training fold?
- Why is macro-F1 useful alongside harmful recall and benign false-positive rate?
- Does the extra hidden layer improve enough to justify its cost in these results?
- What does a positive B − A mean? Why is the ROPE a practical judgment?
- What can a confidently wrong prediction tell us, and what can it not prove?

**Limits:** one balanced sample, one five-fold run, possible related prompts across folds, imperfect labels and uncalibrated confidence. Official test performance and deployed-agent safety are not measured.

**Reuse and AI disclosure:** BPE, the encoder interface, token indicators, `SparseMLP`, sparse batching/training and Bayesian analysis come from the official course repository. The project adds counts, TF-IDF, a second hidden layer and this experiment. OpenAI Codex assisted with implementing and checking the notebook and helpers. The team is responsible for understanding and reviewing them. No new harmful prompts were generated.

## 15. Findings from this run

The cells below summarize the measurements above. They update when the notebook is rerun, so we do not carry conclusions from an older configuration into a new experiment.

The highest observed mean is a **descriptive ranking**, not proof of superiority over all alternatives. The Bayesian statements refer only to our two preselected contrasts, with ROPE ±0.01. Confidence-based errors concern the descriptive leader.

In [ ]:
from IPython.display import Markdown

names = {"logistic": "logistic regression", "mlp1": "one-hidden-layer MLP", "mlp2": "two-hidden-layer MLP"}
representations = {"indicators": "token indicators", "bow": "token counts", "tfidf": "TF-IDF"}
best = summary.loc[summary["macro_f1_mean"].idxmax()]
representation_comparison = bayesian_results.iloc[0]
architecture_comparison = bayesian_results.iloc[1]

print(run_label)
display(summary.pivot(index="model", columns="representation", values="macro_f1_mean")
        .reindex(index=model_names, columns=representation_names).round(4))
display(Markdown(
    f"**Highest observed mean macro-F1:** {names[best['model']]} with {representations[best['representation']]}: "
    f"{best['macro_f1_mean']:.4f} ± {best['macro_f1_std']:.4f} (fold standard deviation).\n\n"
    f"**Representation contrast:** logistic TF-IDF minus logistic indicators = "
    f"{representation_comparison['mean_B_minus_A']:+.4f}. "
    f"The posterior probability of TF-IDF winning by more than the ROPE is "
    f"{representation_comparison['P_B_wins']:.1%}.\n\n"
    f"**Architecture contrast on TF-IDF:** one-layer MLP minus logistic = "
    f"{architecture_comparison['mean_B_minus_A']:+.4f}. "
    f"The posterior probability of practical equivalence is "
    f"{architecture_comparison['P_equivalent']:.1%}.\n\n"
    f"**Errors for the descriptive leader:** {len(errors)} of {len(best_predictions)} held-out predictions were wrong, "
    f"including {int(errors['confidently_wrong'].sum())} with confidence at least 0.8."
))
if use_toy_data:
    print("SMOKE_NOT_SCIENTIFIC: these findings describe artificial topic fixtures only.")

### Interpretation limits

These are five-fold CV results on a balanced sample of the filtered **training** data. The official test remains untouched. A small mean difference does not establish a clear winner, and five overlapping folds provide limited evidence. Confident errors show that probabilities can be misleading; they do not by themselves demonstrate a particular lexical cause or establish deployed-agent safety.